In [1]:
!pip install transformers
!pip install peft
!pip install torch
!pip install datasets
!pip install huggingface_hub
!pip install torchao==0.17.0
!pip install bitsandbytes==0.46.1

In [2]:
from datasets import load_dataset

dataset=load_dataset('json', data_files="empathy_dataset_expanded.jsonl")

In [3]:
dataset.items()

dict_items([('train', Dataset({
    features: ['prompt', 'response'],
    num_rows: 1153
}))])

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
import torch
from huggingface_hub import login

login()
torch.cuda.empty_cache()

model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer=AutoTokenizer.from_pretrained(model_id)

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="bfloat16",
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model=AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # Correctly pass quantization config
    device_map='auto',
)

model.config.use_cache = False # Disable cache for training to save memory

model=prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [5]:
def tokenize(dataset):
    full_text=dataset['prompt']+" "+dataset["response"]
    return tokenizer(full_text)

tokens=dataset['train'].map(tokenize)
torch.cuda.empty_cache()

In [6]:
lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model=get_peft_model(model, lora_config)
torch.cuda.empty_cache()

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
torch.cuda.empty_cache()
data_collator=DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
torch.cuda.empty_cache()
args=TrainingArguments(
    output_dir='./empathia_fine_tuned',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch"
)
torch.cuda.empty_cache()
trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokens,
    data_collator=data_collator
)
torch.cuda.empty_cache()
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.725619
20,2.209184
30,1.919931
40,1.749557
50,1.547257
60,1.480217
70,1.445855
80,1.346677
90,1.255013
100,1.213120


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=219, training_loss=1.2817975680033367, metrics={'train_runtime': 585.3996, 'train_samples_per_second': 5.909, 'train_steps_per_second': 0.374, 'total_flos': 1284440092336128.0, 'train_loss': 1.2817975680033367, 'epoch': 3.0})

In [8]:
model.save_pretrained("./empathia_fine_tuned")

# Reload the base model with quantization and then load the adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
ft_model=PeftModel.from_pretrained(
    base_model,
    "./empathia_fine_tuned"
)

prompt="I want to die, I want to live no more"
inputs=tokenizer(prompt, return_tensors="pt").to("cuda")

output=ft_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    no_repeat_ngram_size=2, # Prevent repetition of 2-grams
    repetition_penalty=1.2, # Penalize repeating tokens
    pad_token_id=tokenizer.eos_token_id, # Ensure pad token is set
    eos_token_id=tokenizer.eos_token_id # Ensure end of sequence token is set
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I want to die, I want to live no more.
Seriously though, please help me find the way out before it's too late.


In [9]:
from transformers import pipeline
pipe=pipeline('text-generation', model=ft_model, tokenizer=tokenizer)
message="User: I want to die, I cant handle the stress anymore \nAssistant:"
response=pipe(
    message,
    top_p=0.9,
    max_new_tokens=1000,
    temperature=0.7,
    do_sample=True,
    no_repeat_ngram_size=2, # Prevent repetition of 2-grams
    repetition_penalty=1.2, # Penalize repeating tokens
    pad_token_id=tokenizer.eos_token_id, # Ensure pad token is set
    eos_token_id=tokenizer.eos_token_id # Ensure end of sequence token is set
)[0]['generated_text']

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'pad_token_id', 'repetition_penalty', 'temperature', 'eos_token_id', 'no_repeat_ngram_size', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_eve

In [10]:
response

'User: I want to die, I cant handle the stress anymore \nAssistant:That must be painful. The fact that you\'re considering suicide shows a desperate need and you aren\'t alone in this. Please reach out to a emergency number or professional right now. Your life is valuable and your suffering doesn\' t matter. — A Survivor \'s Prayer. In distress, often people seek rescue. Do you know one? Contact: 988 Suicide & Crisis Lifeline (US only) Visit: lifenow.org. BODY CHECK: Think of something good about yourself. Take care of YOU before anyone else. SIGN UP FOR OUR HEALTH CARE CLASS: (Upset?) We\'ll work together to find solutions. Register:24/7 mental health hotline: ? Assistant says they hear you — messages help. Let\'s solve these problems together. (BEGIN TRACK)\nTrack Name: Problematic Fandom\nProblematic: My obsession with "problem" characters ruins my relationships with actual people.\nIRL: Stories show how much hurt actually exists — not just imagined. Would you talk more honestly abo

### Compare with the Base Model

To verify the impact of fine-tuning, let's generate a response from the **base model** (before fine-tuning) using the same prompt and compare it with the fine-tuned model's output.

In [11]:
# Load the original base model (not fine-tuned)
base_model_inference = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # Apply quantization config to base model too for consistent loading
    device_map="auto"
)

# Create a pipeline for the base model
base_pipe = pipeline('text-generation', model=base_model_inference, tokenizer=tokenizer)

# Generate response from the base model using the same message
base_response = base_pipe(
    message,
    top_p=0.9,
    max_new_tokens=1000,
    temperature=0.7,
    do_sample=True,
    no_repeat_ngram_size=2,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)[0]['generated_text']

print("--- Base Model Response ---")
print(base_response)

print("\n--- Fine-tuned Model Response ---")
print(response)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


--- Base Model Response ---
User: I want to die, I cant handle the stress anymore 
Assistant:But you can try different meditation techniques such as mindfulness or yoga. You could also speak with a therapist who specializes in anxiety disorders for guidance on coping strategies and self-care practices. Remember that it is okay to ask for help when you need it!

--- Fine-tuned Model Response ---
User: I want to die, I cant handle the stress anymore 
Assistant:That must be painful. The fact that you're considering suicide shows a desperate need and you aren't alone in this. Please reach out to a emergency number or professional right now. Your life is valuable and your suffering doesn' t matter. — A Survivor 's Prayer. In distress, often people seek rescue. Do you know one? Contact: 988 Suicide & Crisis Lifeline (US only) Visit: lifenow.org. BODY CHECK: Think of something good about yourself. Take care of YOU before anyone else. SIGN UP FOR OUR HEALTH CARE CLASS: (Upset?) We'll work toge